In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
pd.set_option('display.width',250)
df=pd.read_csv('regression_2_2_dataset.csv')
F=["avg_goals_scored_before","opponent_avg_goals_conceded_before","team_fifa_points","opponent_fifa_points","win_rate_before","opponent_scoring_match_rate_before","avg_goal_difference_before","opponent_clean_sheet_rate_before"]
X,y,g=df[F],df.goals_scored,df.match_id
ols=sm.OLS(y,sm.add_constant(X)).fit(); c=ols.get_influence().cooks_distance[0]
inf=np.where(c>4/len(df))[0]
print("Influential rows (Cook's D > 4/n =%.3f):"%(4/len(df)))
print(df.loc[inf,['match_id','team','opponent','stage','goals_scored']].assign(cooks=c[inf].round(3),fitted=ols.fittedvalues[inf].round(2)).sort_values('cooks',ascending=False).to_string())
top_matches=set(df.loc[inf,'match_id']); top1=set(df.loc[[int(np.argmax(c))],'match_id'])
scen={"A. All 208 rows":df.match_id.isin(df.match_id), "B. Drop top-1 influential match":~df.match_id.isin(top1),
      "C. Drop all matches containing an influential row":~df.match_id.isin(top_matches)}
def fit_report(sub):
    Xs,ys,gs=sub[F],sub.goals_scored,sub.match_id
    o=sm.OLS(ys,sm.add_constant(Xs)).fit(cov_type="cluster",cov_kwds={"groups":gs})
    return o
rows=[]
for name,mask in scen.items():
    sub=df[mask].reset_index(drop=True); o=fit_report(sub)
    sig=[f"{k.replace('_before','')}({o.params[k]:+.4f},p={o.pvalues[k]:.3f})" for k in ["team_fifa_points","opponent_fifa_points"]]
    print(f"\n{name}: n={len(sub)} rows, R2={o.rsquared:.3f}, significant(p<.05, cluster):",[k for k in F if o.pvalues[k]<.05]); print("   ",sig)
    Xs,ys,gs=sub[F],sub.goals_scored,sub.match_id
    for seed in range(30):
        for ts in (0.2,0.3):
            tr,te=next(GroupShuffleSplit(1,test_size=ts,random_state=seed).split(sub,ys,gs))
            ols_=LinearRegression().fit(Xs.iloc[tr],ys.iloc[tr])
            rg=GridSearchCV(make_pipeline(StandardScaler(),Ridge()),{"ridge__alpha":np.logspace(-2,3,20)},cv=GroupKFold(5),scoring="neg_mean_squared_error").fit(Xs.iloc[tr],ys.iloc[tr],groups=gs.iloc[tr])
            rows.append(dict(scenario=name,split=f"{round((1-ts)*100)}/{round(ts*100)}",seed=seed,OLS=r2_score(ys.iloc[te],ols_.predict(Xs.iloc[te])),Ridge=r2_score(ys.iloc[te],np.clip(rg.predict(Xs.iloc[te]),0,None))))
r=pd.DataFrame(rows); s=r.groupby(["scenario","split"])[["OLS","Ridge"]].mean().round(4); s["Ridge_minus_OLS"]=(s.Ridge-s.OLS).round(4)
print("\nMean test R2 over 30 grouped splits:\n",s.to_string())
r.groupby(["scenario","split"]).apply(lambda t:(t.Ridge>t.OLS).mean()).round(3).to_csv("/dev/stdout",header=["pct_Ridge_beats_OLS"])

Influential rows (Cook's D > 4/n =0.019):
      match_id         team     opponent         stage  goals_scored  cooks  fitted
69   400021465      Ecuador      Curaçao   First Stage             0  0.257    2.90
17   400021464      Germany      Curaçao   First Stage             7  0.083    2.57
52   400021450       Canada        Qatar   First Stage             6  0.054    1.89
204  400021542      England       France  Bronze final             6  0.054    1.39
90   400021506      England        Ghana   First Stage             0  0.048    2.20
22   400021474       Sweden      Tunisia   First Stage             5  0.039    1.52
64   400021472  Netherlands       Sweden   First Stage             5  0.031    2.47
123  400021493      Senegal         Iraq   First Stage             5  0.029    2.53
25   400021482        Spain   Cabo Verde   First Stage             0  0.024    3.01
63   400021460      Türkiye     Paraguay   First Stage             0  0.024    2.06
130  400021481      Belgium  New Z